In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Also add parent directories of the current directory (when running from notebooks/)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 04. Probability and Statistics - The Language of Uncertainty

> **Learning Goals**
> - Understand probability distributions, expectation, variance, and covariance.
> - Interpret Bayes' theorem in code.
> - Connect softmax, likelihood, and maximum likelihood estimation to LLM training.

## 4.1 Probability

Probability quantifies uncertainty. An event $A$ has probability $P(A)$ satisfying:

$$0 \leq P(A) \leq 1, \quad P(\Omega)=1$$

In machine learning, a model output is often interpreted as a probability distribution.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Convergence of coin-flip frequency
rng = np.random.default_rng(42)
n_flips = 10000
flips = rng.binomial(1, 0.5, n_flips)  # 1 = heads
cumulative_freq = np.cumsum(flips) / np.arange(1, n_flips + 1)

plt.figure(figsize=(10, 5))
plt.plot(cumulative_freq, linewidth=1.5)
plt.axhline(0.5, color='r', linestyle='--', label='Theoretical probability = 0.5')
plt.xscale('log')
plt.xlabel('Number of flips (log scale)')
plt.ylabel('Heads frequency')
plt.title('Law of Large Numbers: Frequency converges to probability')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch04_law_of_large_numbers.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Frequency after 10,000 flips: {cumulative_freq[-1]:.4f} (theoretical value 0.5)")


## 4.2 Discrete and Continuous Probability Distributions

- **Discrete distribution**: distribution over countable outcomes, such as dice rolls or tokens
- **Continuous distribution**: distribution over continuous values, such as a normal distribution

LLMs predict the next token as a discrete probability distribution over the vocabulary.


In [ ]:
# Visualize the normal distribution
def normal_pdf(x, mu, sigma):
    return (1 / (np.sqrt(2 * np.pi) * sigma)) * np.exp(-((x - mu)**2) / (2 * sigma**2))

x = np.linspace(-5, 8, 200)
fig, ax = plt.subplots(figsize=(10, 5))
for mu, sigma, label in [(0, 1, 'N(0, 1)'), (2, 1, 'N(2, 1)'), (0, 2, 'N(0, 4)')]:
    ax.plot(x, normal_pdf(x, mu, sigma), linewidth=2, label=label)
ax.set_xlabel('x'); ax.set_ylabel('p(x)')
ax.set_title('Normal distribution PDF')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch04_normal_dist.png', dpi=100, bbox_inches='tight')
plt.show()

# Sample statistics
samples = np.random.default_rng(0).normal(0, 1, 10000)
print(f"N(0,1) Sample 10000 samples:")
print(f"  Mean (Theory 0): {samples.mean():.4f}")
print(f"  Variance (Theory 1): {samples.var():.4f}")
print(f"  Standard Deviation: {samples.std():.4f}")


## 4.3 Bayes' Theorem

$$P(\theta | \mathcal{D}) = \frac{P(\mathcal{D}|\theta)P(\theta)}{P(\mathcal{D})}$$

- $P(\theta)$: prior - belief before observing data
- $P(\mathcal{D} | \theta)$: likelihood - probability of data under $\theta$
- $P(\theta | \mathcal{D})$: posterior - belief after observing data
- $P(\mathcal{D})$: evidence - normalization constant

Maximum likelihood estimation (MLE), commonly used in LLM training, can be viewed as a special case that focuses on the likelihood.


In [ ]:
# Bayes' theorem example: disease test
# P(disease) = 0.01, P(positive|disease) = 0.99, P(positive|healthy) = 0.05
P_D = 0.01
P_notD = 1 - P_D
P_pos_given_D = 0.99
P_pos_given_notD = 0.05

# P(positive) = P(positive|disease)P(disease) + P(positive|healthy)P(healthy)
P_pos = P_pos_given_D * P_D + P_pos_given_notD * P_notD

# Bayes: P(disease|positive) = P(positive|disease)P(disease) / P(positive)
P_D_given_pos = P_pos_given_D * P_D / P_pos

print("Bayes' theorem example: disease test")
print(f"  P(disease) = {P_D}")
print(f"  P(positive|disease) = {P_pos_given_D} (sensitivity)")
print(f"  P(positive|healthy) = {P_pos_given_notD} (false positive rate)")
print(f"  P(positive) = {P_pos:.4f}")
print(f"  P(disease|positive) = {P_D_given_pos:.4f}")
print(f"\n  Even after a positive test, the disease probability is {P_D_given_pos*100:.1f}%!")
print(f"  This is counterintuitive because the disease is rare, so false positives outnumber true positives.")


## 4.4 Expectation, Variance, and Covariance

- **Expectation**: $\mathbb{E}[X] = \sum_x x P(x)$ - average value
- **Variance**: $\mathrm{Var}(X) = \mathbb{E}[(X - \mathbb{E}[X])^2]$ - spread
- **Covariance**: $\mathrm{Cov}(X, Y) = \mathbb{E}[(X-\mathbb{E}[X])(Y-\mathbb{E}[Y])]$
- **Correlation**: $\rho(X, Y) = \frac{\mathrm{Cov}(X, Y)}{\sigma_X \sigma_Y} \in [-1, 1]$


In [ ]:
# Compute expectation and variance
rng = np.random.default_rng(0)

# Two variables
n = 5000
X = rng.normal(0, 1, n)
Y = 0.7 * X + rng.normal(0, 0.5, n)  # Correlated with X

print(f"X: E[X] = {X.mean():.4f} (Theory 0), Var[X] = {X.var():.4f} (Theory 1)")
print(f"Y: E[Y] = {Y.mean():.4f}, Var[Y] = {Y.var():.4f}")
print(f"Cov(X, Y) = {np.cov(X, Y)[0, 1]:.4f}")
print(f"Corr(X, Y) = {np.corrcoef(X, Y)[0, 1]:.4f} (Theory ≈ 0.81)")

# Scatter plot
plt.figure(figsize=(7, 6))
plt.scatter(X, Y, alpha=0.3, s=5)
plt.xlabel('X'); plt.ylabel('Y')
plt.title(f'Covariance/Correlation Visualization (ρ={np.corrcoef(X, Y)[0, 1]:.3f})')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.tight_layout()
plt.savefig('../figures/ch04_correlation.png', dpi=100, bbox_inches='tight')
plt.show()


## 4.5 Maximum Likelihood Estimation (MLE)

Given data $\mathcal{D} = \{x_1, \ldots, x_N\}$, choose the parameter $\theta$ that maximizes the likelihood:

$$\mathcal{L}(\theta) = P(\mathcal{D}|\theta) = \prod_{i=1}^{N} P(x_i|\theta)$$

MLE: $\hat{\theta}_{\text{MLE}} = \arg\max_\theta \mathcal{L}(\theta)$

Log-likelihood:

$$\ell(\theta) = \sum_i \log P(x_i|\theta)$$

In LLM training, the loss function is $-\ell(\theta)$: the negative log-likelihood (NLL), also known as cross-entropy loss.


In [ ]:
# MLE example: estimate normal distribution parameters
rng = np.random.default_rng(42)
true_mu, true_sigma = 3.0, 1.5
data = rng.normal(true_mu, true_sigma, 1000)

# MLE for a normal distribution: mu_hat = mean, sigma_hat = sqrt(var)
mu_mle = data.mean()
sigma_mle = data.std()
print(f"True values: mu={true_mu}, sigma={true_sigma}")
print(f"MLE:  mu_hat={mu_mle:.4f}, sigma_hat={sigma_mle:.4f}")

# Visualize the log-likelihood function (mu only)
def log_likelihood(mu, sigma, data):
    return np.sum(-0.5 * np.log(2 * np.pi * sigma**2) - ((data - mu)**2) / (2 * sigma**2))

mus = np.linspace(2, 4, 100)
lls = [log_likelihood(m, sigma_mle, data) for m in mus]

plt.figure(figsize=(8, 5))
plt.plot(mus, lls, 'b-', linewidth=2)
plt.axvline(mu_mle, color='r', linestyle='--', label=f'MLE mu={mu_mle:.3f}')
plt.axvline(true_mu, color='g', linestyle='--', label=f'True values mu={true_mu}')
plt.xlabel('mu'); plt.ylabel('Log-Likelihood')
plt.title('Normal distribution MLE: log-likelihood function')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch04_mle.png', dpi=100, bbox_inches='tight')
plt.show()


## 4.6 Softmax and Temperature

In LLMs, logits $\mathbf{z} \in \mathbb{R}^{|V|}$ are converted into a probability distribution:

$$\mathrm{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**Temperature $\tau$** controls the sharpness of the distribution:

$$p_i = \frac{e^{z_i/\tau}}{\sum_j e^{z_j/\tau}}$$

- $\tau \to 0$: one token dominates (greedy behavior)
- $\tau = 1$: normal softmax
- $\tau \to \infty$: nearly uniform distribution (random behavior)


In [ ]:
# Softmax and temperature effect
def softmax(z, tau=1.0):
    z = np.asarray(z) / tau
    z = z - z.max()  # Stabilize
    e = np.exp(z)
    return e / e.sum()

# Example logits
logits = np.array([1.0, 2.0, 3.0, 0.5, -1.0])
tokens = ['apple', 'banana', 'orange', 'grape', 'cherry']

print(f"Logits: {logits}")
print()
for tau in [0.5, 1.0, 2.0, 10.0]:
    p = softmax(logits, tau=tau)
    print(f"tau={tau:>4.1f}: {p.round(4)}")
    # entropy
    entropy = -np.sum(p * np.log(p + 1e-12))
    print(f"         entropy = {entropy:.4f}")

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, tau in zip(axes, [0.5, 1.0, 2.0, 10.0]):
    p = softmax(logits, tau=tau)
    ax.bar(tokens, p, color='C0')
    ax.set_title(f'tau={tau}')
    ax.set_ylabel('Probability')
    ax.set_ylim(0, 1)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../figures/ch04_softmax_temperature.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n=> Lower temperature makes the distribution sharper and more deterministic; higher temperature makes it flatter and more random.")


## 4.7 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Expectation | $\mathbb{E}[X] = \sum x P(x)$ | Average value |
| Variance | $\mathrm{Var}(X)=\mathbb{E}[(X-\mathbb{E}[X])^2]$ | Spread |
| Bayes' theorem | $P(\theta\|\mathcal{D}) = \frac{P(\mathcal{D}\|\theta)P(\theta)}{P(\mathcal{D})}$ | Update belief |
| MLE | $\hat\theta = \arg\max P(\mathcal{D}\|\theta)$ | Estimate parameters by likelihood |
| Softmax | $\sigma(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$ | logits to probabilities |
| Temperature | $\sigma(z/\tau)$ | Control distribution sharpness |

### Practice Problems
1. Simulate 1,000 rolls of a fair die and verify that each outcome has probability about $1/6$.
2. Sample 1,000 values from $\mathcal{N}(2,4)$ and estimate $\mu$ and $\sigma$ with MLE.
3. Use Bayes' theorem: a disease has prevalence 30%, a test says "free" with 80% probability for infected people, and the false-free rate is 5%. What is the probability of being infected after a "free" result?
4. Compute softmax distributions for temperatures $\tau=0.1, 1.0, 10$ and explain how $\tau$ changes the distribution.
5. Compare MLE and MAP. Explain how MAP is related to adding an $L^2$ regularization term when the prior is $P(\theta)=\mathcal{N}(0,\lambda)$.
